<a href="https://colab.research.google.com/github/Madison-Barclay/PDAP_Updates/blob/main/ocr_via_groq_with_qwen38_27b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook demonstrates how the Qwen3.8-27b language & vision model can be queried for handwriting transcription. The model is too large for us to have it running on our own machines. So we are using the Groq service (based in France, not that guy down in the States) to explore. You can sign up for a free account at [https://console.groq.com/home](https://console.groq.com/home) then you'll want to create a key by clicking on [keys](https://console.groq.com/keys). Give it a name, and select how long you want the key to be in service. Copy the string of numbers and text that it shows you - this will be the only time you see this string - and then over on the left here in Colab click on the 'key' icon. Click on `add new secret` and type in the name of the key (`GROQ_API_KEY` for example) and then paste the string you copied from Groq into the value box. This will save automatically.

The new few cells load an image of a handwritten entry from a First World War battalion diary. Then, we install some code for working with Groq. We are going to send the image to Groq and have the Qwen model decipher the markings into text. We turn the image into a string of letters and numbers by encoding it using the base 64 system. Then, we send it to Groq and the model with instructions on how to read it.

The second half of this notebook does the same thing, but wraps everything in helper functions to write the results to file, to break a pdf into a folder of images, or to read all the images in a folder and pass them one at a time to the model.

Because we are using the free access to Groq, there are rate limits - we can only pass so much data at a time before we get locked out for a while. The point of this notebook is to show you what can be accomplished; if you were needing serious access to a model, you'd probably have to upgrade your account with Groq.

In [2]:
!wget https://carleton.ca/xlab/wp-content/uploads/en-or-transcribing-handwritten-text-with-python-and-azure-07-400x337.png -O diary.png

--2026-10-19 17:37:47--  https://carleton.ca/xlab/wp-content/uploads/en-or-transcribing-handwritten-text-with-python-and-azure-07-400x337.png
Resolving carleton.ca (carleton.ca)... 134.117.206.206
Connecting to carleton.ca (carleton.ca)|134.117.206.206|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 218286 (213K) [image/png]
Saving to: ‘diary.png’

diary.png           100%[===================>] 213.17K   772KB/s    in 0.3s    

2026-10-19 17:37:47 (772 KB/s) - ‘diary.png’ saved [218286/218286]



In [3]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 2.6 MB/s eta 0:00:00


In [4]:
# this cell should encode the image for passing to the groq client.
import base64

with open("diary.png", "rb") as image_file:
    encoded_string = base64.b64encode(image_file.read()).decode('utf-8')

image = f"data:image/png;base64,{encoded_string}"

In the next bit, the 'system' message gets passed to the model every time you run this. The commands are a bit like spell-casting: you're trying to push the model into that part of its space that understands what handwriting looks like. The 'user' message is the instruction specific to the image that you are going to pass through. Strictly speaking, I don't think we need a user message and you could run this bit with just empty "".

In [11]:
import os
import json
from google.colab import userdata
from groq import Groq

try:
    api_key = userdata.get('groq')
except Exception:
    api_key = os.environ.get('GROQ_API_KEY')

if not api_key:
    raise ValueError("Please ensure the 'groq' secret is set in your Colab Secrets (the key icon in the left panel).")

client = Groq(api_key=api_key)
completion = client.chat.completions.create(
    model="qwen/qwen3.8-27b",
    messages=[
      {
        "role": "system",
        "content": "You provide excellent OCR of any submitted image. You only return the transcribed text, following normal transcription conventions. Return as json."
      },
      {
        "role": "user",
        "content": [
          {
            "type": "text",
            "text": "Please transcribe this handwritten text image."
          },
          {
            "type": "image_url",
            "image_url": {
              "url": image ## this is now the encoded image data
            }
          }
        ]
      }
    ],
    temperature=0.1,
    max_completion_tokens=1000,
    top_p=0.95,
    stream=True,
    stop=None
)

# Accumulate the streamed response
full_response = ""
for chunk in completion:
    content = chunk.choices[0].delta.content or ""
    print(content, end="")
    full_response += content

# Clean markdown wrappers if present in the streamed content
cleaned_response = full_response.strip()
if cleaned_response.startswith("```json"):
    cleaned_response = cleaned_response[7:]
if cleaned_response.endswith("```"):
    cleaned_response = cleaned_response[:-3]
cleaned_response = cleaned_response.strip()

try:
    parsed_json = json.loads(cleaned_response)
    transcribed_text = parsed_json.get("text", cleaned_response)
except Exception:
    transcribed_text = cleaned_response
###------------------ make sure the next line points to the file you want to transcribe
image_filename = "diary.png"
###------------------- that way the resulting transcript gets the appropriate filename header in your output file.

# 1. Append to JSON file
json_file_path = "transcriptions.json"
existing_data = []
if os.path.exists(json_file_path):
    try:
        with open(json_file_path, "r", encoding="utf-8") as jf:
            existing_data = json.load(jf)
            if not isinstance(existing_data, list):
                existing_data = [existing_data]
    except Exception:
        existing_data = []

existing_data.append({
    "image_file": image_filename,
    "transcription": transcribed_text
})

with open(json_file_path, "w", encoding="utf-8") as jf:
    json.dump(existing_data, jf, indent=2, ensure_ascii=False)

# 2. Append to Markdown file
md_file_path = "transcriptions.md"
markdown_entry = f"""
## Transcription for `{image_filename}`

{transcribed_text}

---
"""
with open(md_file_path, "a", encoding="utf-8") as mf:
    mf.write(markdown_entry)

print("\n\nSaved results successfully to transcriptions.json and transcriptions.md")

```json
{
  "text": "DECEMBRE\n28 VENDREDI. Ss Innocents 362-3\nClear and Cold - lovely out.\nVisit - from Mme Thomas &\nfive daughters from\nPontarlier.\nOur doctor, M. Muret,\narrived - good fellow.\nGot orders to go with Capt\nMorrison Vt. Hood and 200\nmen to another part of\nFrance.\nWrote Dyer\nBilled up mess account -\nCash 30 francs for the month"
}
```

Saved results successfully to transcriptions.json and transcriptions.md


Now, we'll wrap all of that up in some helper scripts. We'll need some extra lego bricks to work with pdfs and so on. Then we'll define some functions. The last block shows how to execute the complete pipeline.

In [12]:
!pip install pypdfium
!pip install pdf2image


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 9.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pypdfium: filename=pypdfium-0.0.15-py3-none-any.whl size=7838812 sha256=596e7ce32721bf4ff8765c5687edcc0b7c03b2e9c3b4089a402740e03bd0d68d
  Stored in directory: /root/.cache/pip/wheels/82/8e/ad/47305e1ce677055a493a6b8c562c787d252130c3ed754965e7
Successfully built pypdfium


In [13]:
# Update local package index and then install poppler-utils
!apt-get update -y
!apt-get install -y poppler-utils

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:3 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [75.3 kB]
Get:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.4 MB]
Get:10 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:11 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,549 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,729 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,069 kB]
Get:14 htt

and this next cell has all of our functions altogether.

In [14]:
import os
import base64
import json
import time
from pdf2image import convert_from_path
from google.colab import userdata
from groq import Groq

def pdf_to_images(pdf_path, output_folder):
    """Converts a PDF file into a series of PNG images in a folder."""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    print(f"Converting {pdf_path} to images...")
    # Convert PDF pages to PIL Images
    pages = convert_from_path(pdf_path, dpi=200)

    image_paths = []
    for i, page in enumerate(pages):
        image_name = f"page_{i+1}.png"
        image_path = os.path.join(output_folder, image_name)
        page.save(image_path, "PNG")
        image_paths.append(image_path)
        print(f"Saved: {image_path}")

    return image_paths

def save_transcription_to_files(image_filename, transcribed_text, json_path="transcriptions.json", md_path="transcriptions.md"):
    """Appends transcription results to both a JSON file and a Markdown file."""
    # 1. Append to JSON file
    existing_data = []
    if os.path.exists(json_path):
        try:
            with open(json_path, "r", encoding="utf-8") as jf:
                existing_data = json.load(jf)
                if not isinstance(existing_data, list):
                    existing_data = [existing_data]
        except Exception:
            existing_data = []

    existing_data.append({
        "image_file": image_filename,
        "transcription": transcribed_text
    })

    with open(json_path, "w", encoding="utf-8") as jf:
        json.dump(existing_data, jf, indent=2, ensure_ascii=False)

    # 2. Append to Markdown file
    markdown_entry = f"\n## Transcription for `{image_filename}`\n\n{transcribed_text}\n\n---\n"
    with open(md_path, "a", encoding="utf-8") as mf:
        mf.write(markdown_entry)

    print(f"Saved transcription for {image_filename} to {json_path} and {md_path}")

def ocr_image_folder(image_folder, api_key):
    """Iterates over all PNG images in a folder, performs OCR via Groq, and saves the output with rate-limiting retries."""
    client = Groq(api_key=api_key)

    # Get all PNG images sorted by page number
    images = sorted([
        f for f in os.listdir(image_folder)
        if f.lower().endswith(('.png', '.jpg', '.jpeg'))
    ])

    for img_name in images:
        img_path = os.path.join(image_folder, img_name)
        print(f"Processing OCR for {img_name}...")

        # Encode image to Base64 data URI
        with open(img_path, "rb") as img_file:
            encoded_string = base64.b64encode(img_file.read()).decode('utf-8')
        image_data_uri = f"data:image/png;base64,{encoded_string}"

        # Try to call Groq API with retries
        max_retries = 5
        delay = 10
        success = False

        for attempt in range(max_retries):
            try:
                completion = client.chat.completions.create(
                    model="qwen/qwen3.8-27b",
                    messages=[
                      {
                        "role": "system",
                        "content": "You provide excellent OCR of any submitted image. You only return the transcribed text, following normal transcription conventions. Return as json."
                      },
                      {
                        "role": "user",
                        "content": [
                          {
                            "type": "text",
                            "text": "Please transcribe this handwritten text image."
                          },
                          {
                            "type": "image_url",
                            "image_url": {
                              "url": image_data_uri
                            }
                          }
                        ]
                      }
                    ],
                    temperature=0.1, # you keep this number low to reduce errors
                    max_completion_tokens=1000,  # Lower token limits to avoid rate-limit constraints
                    top_p=0.95, # you keep this number high to only select the most likely next tokens while generating.
                    stream=False
                )

                response_text = completion.choices[0].message.content.strip()

                # Clean up response if wrapped in markdown
                if response_text.startswith("```json"):
                    response_text = response_text[7:]
                if response_text.endswith("```"):
                    response_text = response_text[:-3]
                response_text = response_text.strip()

                try:
                    parsed_json = json.loads(response_text)
                    transcribed_text = parsed_json.get("text", response_text)
                except Exception:
                    transcribed_text = response_text

                print(f"Transcription for {img_name}:\n{transcribed_text}\n")

                # Save the results
                save_transcription_to_files(img_name, transcribed_text)
                success = True
                break

            except Exception as e:
                if "429" in str(e):
                    print(f"Rate limited on {img_name}. Retrying in {delay} seconds (Attempt {attempt+1}/{max_retries})...")
                    time.sleep(delay)
                    delay *= 2
                else:
                    print(f"Error processing {img_name}: {e}")
                    break

        if not success:
            print(f"Failed to process OCR for {img_name} after maximum retries.\n")

In [15]:
## here is some horribly complicated handwritten information,
## some record sheets from an archaeological excavation.
!wget -O 1C20LOWAR_C32070_East-Drawing_Register_1_6.pdf https://archaeologydataservice.ac.uk/archiveDS/archiveDownload?t=arch-5014-1/dissemination/documents/site_records/1C20LOWAR_C32070_East-Drawing_Register_1_6.pdf


--2026-10-19 17:49:53--  https://archaeologydataservice.ac.uk/archiveDS/archiveDownload?t=arch-5014-1/dissemination/documents/site_records/1C20LOWAR_C32070_East-Drawing_Register_1_6.pdf
Resolving archaeologydataservice.ac.uk (archaeologydataservice.ac.uk)... 172.67.177.120, 104.21.75.139, 2606:4700:3030::6815:4b8b, ...
Connecting to archaeologydataservice.ac.uk (archaeologydataservice.ac.uk)|172.67.177.120|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 534102 (522K) [application/pdf]
Saving to: ‘1C20LOWAR_C32070_East-Drawing_Register_1_6.pdf’

1C20LOWAR_C32070_Ea 100%[===================>] 521.58K  1.07MB/s    in 0.5s    

2026-10-19 17:49:54 (1.07 MB/s) - ‘1C20LOWAR_C32070_East-Drawing_Register_1_6.pdf’ saved [534102/534102]



In [16]:
# this will run the full pipeline:
# pdf -> folder of jpgs -> turn each into base 64 and pass each to the vision model -> append the transcription to filename.
# note that the script depends on the `api_key` variable, which we first set back up in one of the first cells, eg

# api_key = os.environ.get('GROQ_API_KEY')

#----

# Execute the full pipeline on the PDF present in your folder
pdf_file = "/content/1C20LOWAR_C32070_East-Drawing_Register_1_6.pdf"
output_img_folder = "/content/pdf_pages_extracted"

# 1. Convert PDF to a folder of images
pdf_to_images(pdf_file, output_img_folder)

# 2. Iterate over the folder and run OCR on each page
ocr_image_folder(output_img_folder, api_key)

Converting /content/1C20LOWAR_C32070_East-Drawing_Register_1_6.pdf to images...
Saved: /content/pdf_pages_extracted/page_1.png
Saved: /content/pdf_pages_extracted/page_2.png
Saved: /content/pdf_pages_extracted/page_3.png
Saved: /content/pdf_pages_extracted/page_4.png
Saved: /content/pdf_pages_extracted/page_5.png
Saved: /content/pdf_pages_extracted/page_6.png
Processing OCR for page_1.png...
Transcription for page_1.png:
{
  "text": "Drawing Register\n\nSite code:\nic2020WAR\nSite Name:\nLower Radbourne\nArea:\nC32C70\n\nDrawing No. | Description | Context No. | Section/ Plan | Drawing Sheet | Issued by: | Date: | Checked\n1001 | Pre-ex plan of F1010, north east corner w/c A | F1010 | P | 1001 | EJR | 26/7/21 |\n1002 | Plan of burnt pit feature, NE of F1010, possible furnace | F1010 | P | 1002 | EJR | 27/7/21 |\n1003 | Plan of yellow deposit and burning of burnt area | (1012) (1018) | P1003 | 1003 | IV | 27/7/21 |\n1004 | Plan of Brown silty clay deposit | (1019) | P | 1004 | WC | 27/7